In [0]:
raw_df = (spark.read
          .format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load("/Volumes/finance_domain_project/default/raw_data/FinanceProjectData.csv"))

In [0]:
raw_df.createOrReplaceTempView("finance_project_data")

In [0]:
spark.sql("select * from finance_project_data")

In [0]:
from pyspark.sql.functions import sha2, concat_ws


In [0]:
new_df = raw_df.withColumn("record_hash_key",sha2(concat_ws("||",*["emp_title","emp_length","home_ownership","annual_inc","zip_code","addr_state","grade","sub_grade","verification_status"]),256))

In [0]:
new_df.printSchema()

In [0]:
new_df.createOrReplaceTempView("new_table")

In [0]:
spark.sql("select count(*) from new_table").show()

In [0]:
spark.sql("select count(distinct(record_hash_key)) from new_table").show()

In [0]:
display(spark.sql("""
    select record_hash_key, count(*) as total_count
    from new_table
    group by record_hash_key
    having total_count > 1
    order by total_count desc
"""))

In [0]:
dbutils.fs.mkdirs("/Volumes/finance_domain_project/default/raw_data/Raw")


In [0]:
spark.sql("""select record_hash_key as member_id, 
emp_title,emp_length,home_ownership,annual_inc,addr_state,zip_code, 'USA' as country,grade,sub_grade,verification_status,tot_hi_cred_lim,application_type,annual_inc_joint,verification_status_joint 
from new_table""").repartition(1).write \
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Raw/customers_data_csv") \
.save()


In [0]:
customers_df = spark.read \
.format("csv") \
.option("inferSchema","true") \
.option("header","true") \
.load("/Volumes/finance_domain_project/default/raw_data/Raw/customers_data_csv")


In [0]:
customers_df.show()

In [0]:
spark.sql("""select id as loan_id, record_hash_key as member_id,
 loan_amnt,funded_amnt,term,int_rate,installment,issue_d, loan_status,purpose,title
from new_table""").repartition(1).write \
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Raw/loans_data_csv") \
.save()


In [0]:
loans_df = spark.read \
.format("csv") \
.option("inferSchema","true") \
.option("header","true") \
.load("/Volumes/finance_domain_project/default/raw_data/Raw/loans_data_csv")


In [0]:
display(loans_df)

In [0]:
spark.sql("""select id as loan_id, total_rec_prncp,total_rec_int,total_rec_late_fee,
total_pymnt,last_pymnt_amnt,last_pymnt_d, next_pymnt_d
from new_table""").repartition(1).write \
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Raw/loans_repayments_csv") \
.save()


In [0]:
loans_repayments_df = spark.read \
.format("csv") \
.option("inferSchema","true") \
.option("header","true") \
.load("/Volumes/finance_domain_project/default/raw_data/Raw/loans_repayments_csv")



In [0]:
spark.sql("""select record_hash_key as member_id, delinq_2yrs,delinq_amnt,pub_rec,pub_rec_bankruptcies,inq_last_6mths,
total_rec_late_fee,mths_since_last_delinq,mths_since_last_record
from new_table""").repartition(1).write \
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Raw/loans_defaulters_csv") \
.save()


In [0]:
loans_defaulters_df = spark.read \
.format("csv") \
.option("inferSchema","true") \
.option("header","true") \
.load("/Volumes/finance_domain_project/default/raw_data/Raw/loans_defaulters_csv")
